In [ ]:
!wget https://storage.googleapis.com/dm_alphamissense/AlphaMissense_hg38.tsv.gz

In [ ]:
# =====================================================================
# ALPHAGENOME CODING-VARIANT VALIDATION PIPELINE (AlphaMissense Track)
# Checkpointed, resumable, memory-safe streaming lookup for Google Colab
# =====================================================================
import numpy as np
import pandas as pd
import os
import json
import gzip
from google.colab import files, drive

# ── 1. Mount Drive for persistent checkpointing & data hosting ───────
drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/alphamissense_coding_pipeline"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'checkpoint_coding.json')
RESULTS_FILE    = os.path.join(CHECKPOINT_DIR, 'results_coding_final.csv')

AM_TSV_PATH = '/content/AlphaMissense_hg38.tsv.gz'

print(f"Checkpoint directory: {CHECKPOINT_DIR}")
if not os.path.exists(AM_TSV_PATH):
    raise FileNotFoundError(f"CRITICAL: AlphaMissense file not found at {AM_TSV_PATH}.")

# ── 2. Constants ─────────────────────────────────────────────────────
CHECKPOINT_EVERY = 20
CODING_BIOTYPES = ['protein_coding']
FEATURE_TYPES_CODING = ['Transcript', 'Exon']

# ── 3. Load and prepare Input VCF/CSVs ───────────────────────────────
print("\nUpload your HIGH background CSV first, then LOW background CSV...")
uploaded = files.upload()
filenames = list(uploaded.keys())

if len(filenames) < 2:
    raise ValueError("Please upload both high and low CSV files.")

dfs = []
for fname in filenames:
    df = pd.read_csv(fname)
    if 'high' in fname.lower():
        df['group'] = 'High-Background'
    elif 'low' in fname.lower():
        df['group'] = 'Low-Background'
    else:
        group = input(f"Is '{fname}' high or low infectivity? (high/low): ").strip().lower()
        df['group'] = 'High-Background' if group == 'high' else 'Low-Background'
    dfs.append(df)
    print(f"  Loaded {fname}: {len(df)} variants → {df['group'].iloc[0]}")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal raw variants loaded: {len(df_all)}")

# ── 4. Filter: coding biotype/feature → SYMBOL → base validity ───────
# Step 4a: coding filter
df_coding = df_all[
    df_all['BIOTYPE'].isin(CODING_BIOTYPES) |
    df_all['Feature_type'].isin(FEATURE_TYPES_CODING)
].copy().reset_index(drop=True)
print(f"After coding variant filter: {len(df_coding)} variants")

# Step 4b: SYMBOL filter (drop blank/NaN)
df_coding = df_coding[
    df_coding['SYMBOL'].notna() & (df_coding['SYMBOL'].str.strip() != '')
].copy().reset_index(drop=True)
print(f"After SYMBOL filter: {len(df_coding)} variants")

if len(df_coding) == 0:
    raise ValueError("No variants found after filtering.")

# ── 5. Derive variant type and expand multi-allelic lines ─────────────
def variant_type(ref, alt):
    if len(alt) > len(ref): return 'Insertion'
    if len(ref) > len(alt): return 'Deletion'
    return 'SNV'

df_coding['type'] = df_coding.apply(lambda r: variant_type(str(r['REF']), str(r['ALT'])), axis=1)

def expand_multiallelic(df):
    rows = []
    for _, row in df.iterrows():
        alts = str(row['ALT']).split(',')
        if len(alts) == 1:
            rows.append(row.to_dict())
        else:
            for alt in alts:
                alt = alt.strip()
                new_row = row.to_dict()
                new_row['ALT'] = alt
                new_row['type'] = variant_type(str(row['REF']), alt)
                rows.append(new_row)
    return pd.DataFrame(rows)

df_coding = expand_multiallelic(df_coding).reset_index(drop=True)

# Generate variant unique tracking keys
df_coding['variant_id'] = (df_coding['CHROM'].astype(str) + '_' +
                           df_coding['POS'].astype(str)   + '_' +
                           df_coding['REF'].astype(str)   + '_' +
                           df_coding['ALT'].astype(str))

df_coding = df_coding.drop_duplicates(subset='variant_id').reset_index(drop=True)

# Step 4c: base validation (after multiallelic expansion so split ALTs are checked)
valid_bases = set('ACGTN')
def is_valid(ref, alt):
    return (set(str(ref)).issubset(valid_bases) and
            set(str(alt)).issubset(valid_bases))

before = len(df_coding)
df_coding = df_coding[
    df_coding.apply(lambda r: is_valid(r['REF'], r['ALT']), axis=1)
].reset_index(drop=True)
print(f"After base validation filter: {len(df_coding)} variants (removed {before - len(df_coding)})")

# ── 6. Memory-Safe AlphaMissense Streaming Lookup ────────────────────
print("\n[STEP] Streaming AlphaMissense TSV dataset...")
your_variant_ids = set(df_coding['variant_id'].tolist())
am_matches = {}

with gzip.open(AM_TSV_PATH, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.strip().split('\t')
        if len(parts) < 10:
            continue

        v_id = f"{parts[0]}_{parts[1]}_{parts[2]}_{parts[3]}"

        if v_id in your_variant_ids:
            try:
                am_matches[v_id] = {
                    'am_pathogenicity': float(parts[8]),  # was 5, correct is 8
                    'am_class': parts[9].strip()          # was 6, correct is 9
                }
            except (ValueError, IndexError):
                continue

print(f"→ Matched {len(am_matches)} out of {len(your_variant_ids)} target variants.")

df_coding['am_pathogenicity'] = df_coding['variant_id'].map(
    lambda x: am_matches.get(x, {}).get('am_pathogenicity', np.nan))
df_coding['am_class'] = df_coding['variant_id'].map(
    lambda x: am_matches.get(x, {}).get('am_class', 'unknown'))

# ── 7. Load/Save Checkpoint Mechanics ────────────────────────────────
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {"processed_ids": [], "results": []}

def save_checkpoint(processed_ids, results):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({"processed_ids": processed_ids, "results": results}, f)
    if results:
        pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)

checkpoint    = load_checkpoint()
processed_ids = set(checkpoint["processed_ids"])
results       = checkpoint["results"]

skipped = len(processed_ids)
if skipped > 0:
    print(f"\n[RESUME] Found checkpoint: {skipped} variants already processed, resuming...")
else:
    print("\n[FRESH] No checkpoint found, starting from scratch...")

# ── 8. Processing and Consensus Generation Loop ───────────────────────
variants_to_run = df_coding[~df_coding['variant_id'].isin(processed_ids)]
total           = len(variants_to_run)
print(f"Variants to process this cycle: {total}\n")

for i, (_, row) in enumerate(variants_to_run.iterrows()):
    v_id  = row['variant_id']
    chrom = str(row['CHROM'])
    pos   = int(row['POS'])
    ref   = str(row['REF'])
    alt   = str(row['ALT'])
    gene  = str(row['SYMBOL'])
    group = row['group']
    vtype = row['type']

    print(f"  [{i+1}/{total}] {gene} {chrom}:{pos} ({vtype}, {group})...", end=' ')

    sift       = row.get('SIFT', np.nan)
    polyphen   = row.get('PolyPhen', np.nan)
    gnomade_af = row.get('gnomADe_AF', 0.0)
    gnomadg_af = row.get('gnomADg_AF', 0.0)
    am_score   = row.get('am_pathogenicity', np.nan)
    am_class   = row.get('am_class', 'unknown')

    is_am_path_flag = 1 if am_class == 'likely_pathogenic' or (pd.notna(am_score) and am_score > 0.564) else 0
    is_sift_flag    = 1 if 'deleterious' in str(sift).lower() else 0
    is_poly_flag    = 1 if 'damaging' in str(polyphen).lower() else 0
    consensus_score = is_am_path_flag + is_sift_flag + is_poly_flag

    record = {
        "variant_id":           v_id,
        "gene":                 gene,
        "group":                group,
        "coordinates":          f"{chrom}:{pos}",
        "type":                 vtype,
        "exon":                 row.get('EXON', 'N/A'),
        "HGVS":                 row.get('HGVSp', 'N/A'),
        "log2FC_expression":    row.get('log2FoldChange', np.nan),
        "padj_expression":      row.get('padj', np.nan),
        "gnomADe_AF":           gnomade_af,
        "gnomADg_AF":           gnomadg_af,
        "SIFT_score":           sift,
        "PolyPhen_score":       polyphen,
        "AlphaMissense_score":  am_score,
        "AlphaMissense_class":  am_class,
        "Consensus_Risk_Score": consensus_score
    }

    results.append(record)
    processed_ids.add(v_id)
    print(f"αM_class={am_class} | Consensus Risk={consensus_score}/3")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(list(processed_ids), results)
        print(f"    [CHECKPOINT] {len(processed_ids)} variants saved to Drive.")

save_checkpoint(list(processed_ids), results)
print(f"\n[SUCCESS] Complete. Output at: {RESULTS_FILE}")